# 2. Frameworks and Zero Trust — Architecture Design

SC-100 expects you to **map a requirement to the right Microsoft framework** and then pick the right product. This notebook:

1. Introduces the four frameworks (MCRA, MCSB, CAF, WAF).
2. Walks through Zero Trust as an architecture pattern with a **bad → best** evolution.
3. Ends with an auto-graded scenario quiz (no interactive input needed).

| Framework | What it gives you | When the exam uses it |
|---|---|---|
| **MCRA** – Microsoft Cybersecurity Reference Architectures | Visual diagrams of how Microsoft security products fit together | *"How do these services connect?"* |
| **MCSB** – Microsoft Cloud Security Benchmark | Control framework with concrete recommendations | *"What controls should we implement?"* |
| **CAF** – Cloud Adoption Framework | Governance & landing-zone methodology | *"How should we structure our Azure environment?"* |
| **WAF** – Azure Well-Architected Framework | Five pillars for workload design: Reliability, Security, Cost Optimization, Operational Excellence, Performance Efficiency | *"Is this workload well-designed?"* |

> ⚠️ **"WAF" is overloaded.** In this notebook **WAF = Well-Architected Framework**. Elsewhere in Azure, **WAF = Web Application Firewall** (the L7 OWASP filter on Application Gateway / Front Door). SC-100 uses both meanings in the same exam. Judge from context: *pillars / workload review* → framework; *OWASP / SQLi / bot protection* → firewall.

## Setup — run this once

This lab uses only the Python standard library (no extra dependencies).

1. Open a terminal in `security-certs/sc-100/01-best-practices/`.
2. Run `uv sync` to create the lab's `.venv`.
3. In VS Code, click the kernel picker (top-right of this notebook) and select **.venv (Python)**.
4. If the kernel doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

Then run the cell below to confirm Python is working.

In [ ]:
import sys, platform
print('Python :', sys.version.split()[0])
print('Kernel :', sys.executable)
print('OS     :', platform.system(), platform.release())
print('Ready to design secure architectures ✅')

## Zero Trust in one sentence

> **Never trust, always verify — and assume breach.**

The three principles you'll see on the exam:

1. **Verify explicitly** — authenticate and authorize every request using all available signals (user, device, location, risk).
2. **Use least-privilege access** — Just-in-Time (PIM), Just-Enough-Access (JEA), risk-based adaptive policies.
3. **Assume breach** — segment, encrypt end-to-end, use analytics to detect lateral movement.

## Bad → Best — a perimeter design evolves toward Zero Trust

Watch how the same requirement ("an employee accesses SharePoint from their laptop") gets more and more Zero-Trust as we tighten it.

In [ ]:
designs = [
    ('❌ CLASSIC PERIMETER',
     'Inside-the-VPN = trusted. Username+password = access. Flat network.',
     ['VPN', 'AD password']),

    ('⚠️ MFA BOLT-ON',
     'VPN still implies trust, but admins require MFA. Users do not.',
     ['VPN', 'MFA (admins only)']),

    ('🟡 IDENTITY-CENTRIC',
     'No VPN for SaaS. MFA for everyone. Conditional Access rules on location.',
     ['Entra ID', 'MFA', 'Conditional Access']),

    ('✅ ZERO TRUST',
     'Every request is evaluated on: user risk + device compliance + app sensitivity + session risk.',
     ['Entra ID P2', 'Intune compliant device', 'Identity Protection', 'Defender for Cloud Apps']),
]
for label, desc, controls in designs:
    print(f'{label}')
    print(f'   behaviour: {desc}')
    print(f'   controls : {", ".join(controls)}')
    print()

## Prioritising Zero Trust work — the Zero Trust adoption framework

Microsoft's guidance for the **order** in which to do Zero Trust work is now published as the **Zero Trust adoption framework** (business scenarios, each with a define-strategy → plan → ready → adopt → govern cycle). The current SC-100 study guide names it explicitly: *"Design solutions that align with the Zero Trust adoption framework."*

> 🗓️ You will still meet the older name **RaMP (Rapid Modernization Plan)** in books, courseware and older practice tests. It is the same idea — a prioritised, phased list of Zero Trust initiatives — under the previous branding. Recognise both; write "Zero Trust adoption framework" when you have the choice.

Exam tip: whichever name the question uses, when it asks *"what should be implemented first?"*, **identity wins**.

In [ ]:
ZERO_TRUST_RAMP = [
    ('1. Verify users explicitly',  'Highest',
        ['MFA for all users', 'Conditional Access + device compliance',
         'Identity Protection risk policies', 'Block legacy auth'],
        ['Entra ID P2', 'Conditional Access', 'Identity Protection']),
    ('2. Verify devices explicitly', 'High',
        ['Intune enrollment', 'Compliance policies', 'Require compliant device in CA',
         'Defender for Endpoint'],
        ['Intune', 'Defender for Endpoint']),
    ('3. Protect data',              'High',
        ['Classify & label (Purview)', 'DLP for email + SharePoint',
         'Encrypt at rest / in transit', 'Auto-labeling'],
        ['Purview IP', 'Purview DLP', 'Key Vault']),
    ('4. Secure infrastructure',     'Medium',
        ['Network segmentation', 'Azure Firewall', 'Private Endpoints',
         'Defender for Cloud CSPM'],
        ['Azure Firewall', 'Private Link', 'Defender for Cloud']),
    ('5. Secure applications',       'Medium',
        ['Managed identities (no stored secrets)', 'Web Application Firewall for web apps',
         'API Management + OAuth', 'DevSecOps'],
        ['Managed Identity', 'Azure WAF', 'APIM', 'GitHub Advanced Security']),
]

icon = {'Highest': '🔴', 'High': '🟠', 'Medium': '🟡'}
for name, pri, actions, products in ZERO_TRUST_RAMP:
    print(f'{icon[pri]} {name} [{pri}]')
    for a in actions: print(f'   ☐ {a}')
    print(f'   products: {", ".join(products)}\n')

## MCSB — Microsoft Cloud Security Benchmark

MCSB is a **checklist of controls** grouped into 12 families. The exam often says *"which MCSB family covers X?"* or *"which control would you apply?"*.

In [ ]:
MCSB = [
    ('NS', 'Network Security',            'NSGs, Azure Firewall, Private Endpoints'),
    ('IM', 'Identity Management',         'Entra ID, Conditional Access'),
    ('PA', 'Privileged Access',           'PIM, PAW, break-glass accounts'),
    ('DP', 'Data Protection',             'Purview, sensitivity labels'),
    ('AM', 'Asset Management',            'Defender for Cloud inventory'),
    ('LT', 'Logging & Threat Detection',  'Sentinel, Defender XDR'),
    ('IR', 'Incident Response',           'Sentinel playbooks, Defender XDR'),
    ('PV', 'Posture & Vulnerability',     'Azure Policy, Defender CSPM'),
    ('ES', 'Endpoint Security',           'Defender for Endpoint'),
    ('BR', 'Backup & Recovery',           'Azure Backup, ASR'),
    ('DS', 'DevOps Security',             'GitHub Advanced Security, Defender for Cloud DevOps security'),
    ('GS', 'Governance & Strategy',       'Azure Policy, Management Groups'),
]
for fid, name, svc in MCSB:
    print(f'[{fid}] {name:<26} → {svc}')

## CAF Landing Zones — the secure foundation for Azure

CAF's **landing zones** give you a pre-wired hierarchy of management groups, subscriptions, networking, identity and policy. New workloads drop into a landing zone and inherit the guardrails.

```
Root Management Group
├── Platform
│   ├── Identity      (Entra Connect, Domain Controllers)
│   ├── Management    (Log Analytics, Automation, Monitoring)
│   └── Connectivity  (Hub VNet, Azure Firewall, ExpressRoute / VPN)
├── Landing Zones
│   ├── Corp    (internal apps, connected to hub)
│   └── Online  (internet-facing apps + WAF + Front Door)
├── Sandbox   (dev / test, isolated)
└── Decommissioned
```

| Level | Controls applied |
|---|---|
| Root MG | Azure Policy: deny public storage, require tags, MCSB initiative |
| Platform – Identity | PIM, break-glass, Conditional Access, Entra Connect hardening |
| Platform – Connectivity | Hub firewall, DDoS, DNS, private peering |
| LZ – Corp | Private Endpoints, VNet peering to hub, NSGs |
| LZ – Online | Azure WAF, Front Door, DDoS Network Protection, public IP restrictions |

## Auto-graded scenario quiz

The cell below lets you edit `MY_ANSWERS` and re-run to score yourself. No interactive input, so it's safe to re-run headlessly.

In [ ]:
SCENARIOS = [
    {'id': 'Q1',
     'question': 'A financial company needs to prove compliance with multiple regulatory standards across Azure AND AWS.',
     'answer': 'MCSB',
     'why': 'Defender for Cloud + MCSB map to regulatory standards (PCI DSS, SOC 2, ISO) and work multi-cloud.'},
    {'id': 'Q2',
     'question': 'Contoso is migrating to Azure and needs a secure foundational environment for 10 teams.',
     'answer': 'CAF',
     'why': 'CAF landing zones give management groups, policies, networking and identity out of the box.'},
    {'id': 'Q3',
     'question': 'An architect wants a picture of how Defender for Endpoint, Sentinel, and Entra ID fit together.',
     'answer': 'MCRA',
     'why': 'MCRA is the diagram set showing how Microsoft security products integrate.'},
    {'id': 'Q4',
     'question': 'A team building a new web application needs to review its security design.',
     'answer': 'WAF',
     'why': 'WAF Security pillar gives workload-specific guidance: threat modeling, defense in depth, etc.'},
    {'id': 'Q5',
     'question': 'The CISO wants the fastest prioritized path to implement Zero Trust.',
     'answer': 'ZERO TRUST ADOPTION FRAMEWORK',
     'why': 'Microsoft\'s prioritized, phased Zero Trust guidance is the Zero Trust adoption framework '
            '(identity -> device -> data -> infrastructure -> apps). Older material calls the same thing '
            'the Rapid Modernization Plan (RaMP) - accept either name if a question uses it.'},
]

# 👇 edit these to your guesses, then re-run the cell
MY_ANSWERS = {'Q1': 'MCSB', 'Q2': 'CAF', 'Q3': 'MCRA', 'Q4': 'WAF', 'Q5': 'Zero Trust adoption framework'}

score = 0
for s in SCENARIOS:
    mine = MY_ANSWERS.get(s['id'], '').strip().upper()
    correct = s['answer'].upper()
    # accept the legacy name for Q5 so older study material still grades correctly
    ok = mine == correct or (s['id'] == 'Q5' and mine in ('RAMP', 'ZERO TRUST RAMP'))
    score += ok
    mark = '✅' if ok else '❌'
    print(f'{mark} {s["id"]}: {s["question"]}')
    print(f'    your answer: {mine or "(blank)"} | correct: {s["answer"]}')
    print(f'    why: {s["why"]}\n')
print(f'Score: {score}/{len(SCENARIOS)}')

## Cheat sheet

| Framework | Use when | Deliverable |
|---|---|---|
| **MCRA** | Need to see how products connect | Architecture diagrams |
| **MCSB** | Need specific security controls | Control checklist with Azure mapping |
| **CAF** | Setting up Azure governance | Landing-zone architecture |
| **WAF** | Designing a single workload | Pillar-based review |
| **Zero Trust adoption framework** (was: RaMP) | Need a phased Zero Trust plan | Prioritized action list |

**Next:** [Notebook 3 — DevSecOps and Shared Responsibility](03_devsecops_and_shared_responsibility.ipynb)